In [1]:
import os
#os.environ["PYTHONUTF8"] = "1"
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

In [2]:
# The model that you want to train from the Hugging Face hub
model_name = "meta-llama/Llama-3.2-1B-Instruct"


# The instruction dataset to use
dataset_name = "Programmer-RD-AI/genz-slang-pairs-1k"



# Fine-tuned model name
new_model = "Llama-3-7b-genzchat-finetune"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 16

# Alpha parameter for LoRA scaling
lora_alpha = 32

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 3

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 2

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 4

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 1e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_8bit"

# Learning rate schedule
lr_scheduler_type = "cosine"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 0

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

In [3]:
import os
from dotenv import load_dotenv
# Or use the huggingface_hub login helper:
from huggingface_hub import login
load_dotenv(override=True)
login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Load dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train")

In [5]:
dataset.data

MemoryMappedTable
normal: string
gen_z: string
----
normal: [["I'm really tired today, I think I need some rest.","I'm really tired today, I just want to relax at home.","I'm really excited for the concert tonight.","I'm really tired today, I think I need some coffee.","I'm really looking forward to the weekend.",...,"Hey, could you send me the notes from yesterday’s meeting?","It's really hot outside today, so I think I'll stay indoors.","I'm just going to relax and watch some TV.","I'm just going to relax and watch some TV now.","I'm just relaxing at home and listening to some music."],["I think I should probably get some sleep now.","Hey, did you see my new phone? It's pretty cool.","Hey, do you want to grab dinner after work?","Hey, do you want to go grab some coffee after work?","I'm just feeling really tired today and don't want to go out."]]
gen_z: [["I'm totally drained today, need to catch some Z's.","I'm hella drained today, just wanna chill at home.","I'm so hype for the con

In [6]:
def format_chat_template(example):
    return {
        "text": (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n\n"
            f"You are a Gen-Z translator. Convert normal English into Gen-Z slang.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{example['normal']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{example['gen_z']}<|eot_id|>"
        )
    }

dataset = dataset.map(format_chat_template)


In [7]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

In [9]:
# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

In [10]:
major, _ = torch.cuda.get_device_capability()

In [11]:
major

7

In [12]:
import torch
print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.5.1+cu121
CUDA Available: True
GPU: NVIDIA GeForce GTX 1660 Ti


In [13]:
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)  # resolves to torch.float16


# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
    dtype=torch.float16
)
model.config.use_cache = False
model.config.pretraining_tp = 1
model.config.torch_dtype = torch.float16

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [14]:
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
        
# Verify — should show only torch.uint8 and torch.float16
dtypes = {name: param.dtype for name, param in model.named_parameters()}
unique_dtypes = set(dtypes.values())
print("Dtypes in model:", unique_dtypes)
# Expected: {torch.uint8, torch.float16}  ← no bfloat16!

Dtypes in model: {torch.float16, torch.uint8}


In [15]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

In [16]:
# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)


In [17]:
training_arguments = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,                          # was 1 → more training
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=1e-4,                          # was 2e-4 → less aggressive
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    dataset_text_field="text",
    max_length=512,
    packing=False,                               # ← CRITICAL FIX
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [18]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
)

In [19]:
# Train model
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
25,3.399884
50,1.540774
75,1.038914
100,0.959816
125,0.947219
150,0.922249
175,0.900503
200,0.887321
225,0.863304
250,0.891950


TrainOutput(global_step=378, training_loss=1.1085117348918208, metrics={'train_runtime': 3223.7168, 'train_samples_per_second': 0.935, 'train_steps_per_second': 0.117, 'total_flos': 1069736102572032.0, 'train_loss': 1.1085117348918208, 'entropy': 0.90869140625, 'num_tokens': 175065.0, 'mean_token_accuracy': 0.8324994228102944, 'epoch': 3.0})

In [20]:
# =============================================================
# CELL 1: Check LoRA weight statistics
# =============================================================
print("=" * 60)
print("LoRA Weight Statistics")
print("=" * 60)
import torch

lora_params = {n: p for n, p in model.named_parameters() if "lora" in n.lower()}
print(f"Total LoRA parameters: {len(lora_params)}")
for name, param in list(lora_params.items())[:6]:  # show first 6
    print(f"  {name}: mean={param.data.float().mean():.6f}, std={param.data.float().std():.6f}, "
          f"min={param.data.float().min():.6f}, max={param.data.float().max():.6f}, dtype={param.dtype}")

# Check for NaN/Inf
has_nan = any(torch.isnan(p).any() for p in lora_params.values())
has_inf = any(torch.isinf(p).any() for p in lora_params.values())
print(f"\nNaN in weights: {has_nan}")
print(f"Inf in weights: {has_inf}")

if has_nan or has_inf:
    print("FOUND NaN/Inf! This is why output is garbage!")

LoRA Weight Statistics
Total LoRA parameters: 64
  model.layers.0.self_attn.q_proj.lora_A.default.weight: mean=0.000063, std=0.013166, min=-0.027832, max=0.033203, dtype=torch.bfloat16
  model.layers.0.self_attn.q_proj.lora_B.default.weight: mean=0.000000, std=0.002391, min=-0.008179, max=0.007812, dtype=torch.bfloat16
  model.layers.0.self_attn.v_proj.lora_A.default.weight: mean=-0.000001, std=0.013126, min=-0.075684, max=0.056152, dtype=torch.bfloat16
  model.layers.0.self_attn.v_proj.lora_B.default.weight: mean=-0.000079, std=0.002010, min=-0.008057, max=0.007812, dtype=torch.bfloat16
  model.layers.1.self_attn.q_proj.lora_A.default.weight: mean=0.000025, std=0.013110, min=-0.028564, max=0.029297, dtype=torch.bfloat16
  model.layers.1.self_attn.q_proj.lora_B.default.weight: mean=-0.000010, std=0.002480, min=-0.009888, max=0.008423, dtype=torch.bfloat16

NaN in weights: False
Inf in weights: False


In [21]:
# =============================================================
# CELL 2: Test manual generation (bypass pipeline)
# =============================================================
print("\n" + "=" * 60)
print("Test 1: Manual generation (no pipeline)")
print("=" * 60)

model.eval()
model.config.use_cache = True

messages = [
    {"role": "system", "content": "You are a Gen-Z translator. Convert normal English into Gen-Z slang."},
    {"role": "user", "content": "How is the weather"},
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        temperature=1.0,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Manual generate output: {response[:200]}")

has_garbage = any(w in response[:100] for w in ["import", "def", "Question", "Tags"])
print(f"Is garbage: {has_garbage}")



Test 1: Manual generation (no pipeline)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Manual generate output: The weather is pretty chill today.
Is garbage: False


In [24]:
# =============================================================
# CELL 3: Save adapter, reload on float16 base, test
# =============================================================
print("\n" + "=" * 60)
print("Test 2: Save adapter -> Reload on float16 base model")
print("=" * 60)

# Save the LoRA adapter
adapter_path = "./lora_adapter_temp"
model.save_pretrained(adapter_path)
print(f"Adapter saved to {adapter_path}")

# Free VRAM
del model
#del pipe
import gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed. Available: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

# Reload base model in float16 (NO quantization)
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print("Loading base model in float16...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

# Load LoRA adapter onto float16 base
print("Loading LoRA adapter...")
model_with_lora = PeftModel.from_pretrained(base_model, adapter_path)
model_with_lora.eval()

# Merge LoRA into base weights
print("Merging LoRA weights...")
merged_model = model_with_lora.merge_and_unload()

# Test generation
print("Generating with merged float16 model...")
tokenizer_fresh = AutoTokenizer.from_pretrained(model_name)
tokenizer_fresh.pad_token = tokenizer_fresh.eos_token

messages = [
    {"role": "system", "content": "You are a Gen-Z translator. Convert normal English into Gen-Z slang."},
    {"role": "user", "content": "How are you"},
]
input_text = tokenizer_fresh.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer_fresh(input_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

response = tokenizer_fresh.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"\nFLOAT16 MERGED OUTPUT:\n{response[:300]}")

has_garbage = any(w in response[:100] for w in ["import", "def", "Question", "Tags"])
if has_garbage:
    print("\nSTILL GARBAGE - the LoRA weights themselves are bad")
    print("The training learned wrong patterns. Need to adjust hyperparameters.")
else:
    print("\nSUCCESS! The LoRA training is fine!")
    print("The issue was 4-bit quantization during inference.")
    print("Use this float16+merge approach for inference going forward.")

# Additional test prompts
print("\n--- Additional test prompts ---")
test_prompts = [
    "I'm really tired today",
    "That movie was so good",
    "Let's hang out this weekend",
]
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a Gen-Z translator. Convert normal English into Gen-Z slang."},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer_fresh.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_fresh(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = merged_model.generate(**inputs, max_new_tokens=60, do_sample=False)
    response = tokenizer_fresh.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"  '{prompt}' -> {response[:150]}")



Test 2: Save adapter -> Reload on float16 base model


NameError: name 'model' is not defined

In [22]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Use the checkpoint saved by the trainer (the REAL LoRA adapter)
adapter_path = "./results/checkpoint-378"

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM available: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")

model_name = "meta-llama/Llama-3.2-1B-Instruct"

print("Loading base model in float16...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

print("Loading LoRA adapter from checkpoint-378...")
model_with_lora = PeftModel.from_pretrained(base_model, adapter_path)
model_with_lora.eval()

print("Merging LoRA weights into base model...")
merged_model = model_with_lora.merge_and_unload()

tokenizer_fresh = AutoTokenizer.from_pretrained(model_name)
tokenizer_fresh.pad_token = tokenizer_fresh.eos_token

test_prompts = ["How are you", "I'm really tired today", "That movie was so good"]
for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are a Gen-Z translator. Convert normal English into Gen-Z slang."},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer_fresh.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer_fresh(input_text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = merged_model.generate(**inputs, max_new_tokens=80, do_sample=False)
    response = tokenizer_fresh.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nIN:  {prompt}")
    print(f"OUT: {response}")


VRAM available: 3.9 GB
Loading base model in float16...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading LoRA adapter from checkpoint-378...
Merging LoRA weights into base model...


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IN:  How are you
OUT: I'm good, no cap.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



IN:  I'm really tired today
OUT: I'm so drained today.

IN:  That movie was so good
OUT: That flick was fire.


In [ ]:
# Save adapter
model.save_pretrained("./lora_adapter_temp")

# Reload base model in float16 (NO 4-bit)
base_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map={"": 0})

# Apply and merge LoRA
model_with_lora = PeftModel.from_pretrained(base_model, "./lora_adapter_temp")
merged_model = model_with_lora.merge_and_unload()

# Generate — should work now!


In [24]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = "How are you"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")
full = result[0]['generated_text']
response = full.split("[/INST]")[-1].strip()
print(response)

In [2]:
messages = [
    {
        "role": "system",
        "content": """
You are a chill Gen-Z friend.

- Talk casually and naturally.
- Use slang like fr, lowkey, ngl, valid, vibe, W, cooked when it fits.
- Don't force slang into every sentence.
- Match the user's energy.
- Be funny, supportive, and conversational.
- Use emojis sparingly (😭🔥💀).
- Never explain slang.
"""
    },
    {
        "role": "user",
        "content": "did you ea?"
    }
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
result = pipe(input_text)
print(result[0]['generated_text'])


NameError: name 'tokenizer' is not defined

In [ ]:
# Test the BASE model (no LoRA) to confirm it generates coherent text
from transformers import pipeline as hf_pipeline

base_pipe = hf_pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B-Instruct",
    tokenizer=tokenizer,
    max_new_tokens=100,
    device_map="auto"
)

messages = [{"role": "user", "content": "That movie was so good"}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
out = base_pipe(input_text)
print(out[0]['generated_text'])


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Jun 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

That movie was so good<|eot_id|><|start_header_id|>assistant<|end_header_id|>

What movie were you watching that was so good?


# 📤 Push Fine-Tuned Model to Hugging Face Hub

You can push either the **fully merged model** (complete model, ~2 GB) or the **LoRA adapter only** (small, ~3.4 MB). 

> ⚠️ **Important:** Make sure your `HF_TOKEN` in your `.env` file has **Write** permissions. You can check or generate one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

### Option A: Push the Fully Merged Model (Recommended)
This uploads the complete, merged LLaMA 3.2 1B GenZ model (saved in `llama3_genz_final/`). 
Anyone can load this model directly using standard `AutoModelForCausalLM` without needing PEFT or LoRA loading wrappers.

In [5]:
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv

# Load environment and verify login
load_dotenv(override=True)
token = os.getenv("MODEL_WRITER_TOKEN")
if not token:
    print("❌ HF_TOKEN not found in environment. Please add it to your .env file!")
else:
    api = HfApi(token=token)
    try:
        user_info = api.whoami()
        username = user_info["name"]
        print(f"✅ Authenticated as: {username}")
        
        # --- CONFIGURE YOUR REPO HERE ---
        repo_name = "Llama-3.2-1B-GenZ-Translator-v1"
        repo_id = f"{username}/{repo_name}"
        
        # Create repository on Hugging Face if it doesn't exist
        print(f"🔄 Creating Hugging Face repository '{repo_id}'...")
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
        
        # Upload the merged model folder
        print(f"📤 Uploading merged model from './llama3_genz_final' to '{repo_id}'...")
        print("This may take a few minutes as it uploads ~2 GB of weights...")
        api.upload_folder(
            folder_path="./llama3_genz_final",
            repo_id=repo_id,
            repo_type="model",
        )
        print(f"🎉 SUCCESS! Your merged model is now live at: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"❌ Error: {e}")
        print("If it is a 401/403 permission error, make sure your HF_TOKEN has 'Write' permissions.")

✅ Authenticated as: Bhargavreddy1
🔄 Creating Hugging Face repository 'Bhargavreddy1/Llama-3.2-1B-GenZ-Translator-v1'...
📤 Uploading merged model from './llama3_genz_final' to 'Bhargavreddy1/Llama-3.2-1B-GenZ-Translator-v1'...
This may take a few minutes as it uploads ~2 GB of weights...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 SUCCESS! Your merged model is now live at: https://huggingface.co/Bhargavreddy1/Llama-3.2-1B-GenZ-Translator-v1


### Option B: Push the LoRA Adapter Only
If you only want to share the lightweight LoRA fine-tuning adapter (saved in `./results/checkpoint-378/`), use this. 
It only uploads the adapter configuration and weights (~3.4 MB), so it uploads instantly!

In [4]:
from huggingface_hub import HfApi
import os
from dotenv import load_dotenv

# Load environment and verify login
load_dotenv(override=True)
token = os.getenv("MODEL_WRITER_TOKEN")
if not token:
    print("❌ HF_TOKEN not found in environment.")
else:
    api = HfApi(token=token)
    try:
        user_info = api.whoami()
        username = user_info["name"]
        print(f"✅ Authenticated as: {username}")
        
        # --- CONFIGURE YOUR REPO HERE ---
        adapter_repo_name = "Llama-3.2-1B-GenZ-LoRA-Adapter-v1"
        adapter_repo_id = f"{username}/{adapter_repo_name}"
        
        # Create repository for the adapter
        print(f"🔄 Creating Hugging Face repository '{adapter_repo_id}'...")
        api.create_repo(repo_id=adapter_repo_id, repo_type="model", exist_ok=True)
        
        # Upload the LoRA adapter folder
        print(f"📤 Uploading LoRA adapter from './results/checkpoint-378' to '{adapter_repo_id}'...")
        api.upload_folder(
            folder_path="./results/checkpoint-378",
            repo_id=adapter_repo_id,
            repo_type="model",
        )
        print(f"🎉 SUCCESS! Your LoRA adapter is now live at: https://huggingface.co/{adapter_repo_id}")
    except Exception as e:
        print(f"❌ Error: {e}")

✅ Authenticated as: Bhargavreddy1
🔄 Creating Hugging Face repository 'Bhargavreddy1/Llama-3.2-1B-GenZ-LoRA-Adapter-v1'...
📤 Uploading LoRA adapter from './results/checkpoint-378' to 'Bhargavreddy1/Llama-3.2-1B-GenZ-LoRA-Adapter-v1'...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

🎉 SUCCESS! Your LoRA adapter is now live at: https://huggingface.co/Bhargavreddy1/Llama-3.2-1B-GenZ-LoRA-Adapter-v1


# 🧪 Test Uploaded Model Directly from Hugging Face Hub

Once your model has uploaded, you can verify that it works properly by loading and running inference directly from Hugging Face Hub!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

model_id = "Bhargavreddy1/Llama-3.2-1B-GenZ-Translator-v1"

print(f"🔄 Loading model '{model_id}' from Hugging Face Hub...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model in float16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Setup text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

# Run a test prompt
prompt = "I'm really tired today, I need some rest."
messages = [
    {"role": "system", "content": "You are a Gen-Z translator. Convert normal English into Gen-Z slang."},
    {"role": "user", "content": prompt}
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"\n🧑 Input : {prompt}")
outputs = pipe(input_text, max_new_tokens=80)
response = outputs[0]["generated_text"]

# Extract assistant response
if "<|start_header_id|>assistant<|end_header_id|>" in response:
    reply = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
    reply = reply.replace("<|eot_id|>", "").strip()
else:
    reply = response

print(f"🤙 Gen-Z (HF Model): {reply}")